# Lab 1.1 &mdash; From a Stateless Call to an Agent Loop

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 25 min &nbsp;|&nbsp; **Day 1 &middot; Module 1 &mdash; Agents vs. Multi-Agent Systems**

### What you'll do
- Prove to yourself that a model call carries nothing from the call before it
- Build the loop by hand -- decide, act, observe, and above all *stop*
- Add loop detection, the failure that quietly burns a budget in production

> **How this lab works.** Fill every `___`, then run the **Self-check** cell under each section.
> Graded cells are plain Python and never call a model, so your score never depends on a
> live endpoint. Cells marked **Run it for real** do call the sandbox model; if it is not
> reachable they print how to fix it instead of crashing.

> **The thread.** All five Module 1 labs work one case: payment exceptions on a small
> synthetic ledger. What you build here is extended in every later lab.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-1-01")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError:
        print("(a blank above is still unfilled -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These two values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
LLM_MODEL    = os.environ.get("LAB_LLM_MODEL")    or os.environ.get("OPENAI_MODEL")
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm = None
def get_llm(temperature: float = 0.0):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    global _llm
    if _llm is None:
        from langchain_openai import ChatOpenAI
        _llm = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                          api_key=LLM_API_KEY, temperature=temperature)
    return _llm

def ask(prompt: str, system: str | None = None) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm().invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- graded cells still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# One domain runs through all five Module 1 labs: payment exceptions on a small ledger.
# Nothing here is real data and nothing leaves this notebook.

LEDGER = {
    "PMT-1001": {"amount": 250000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "settled",  "value_date": "2026-09-01", "reason_code": None},
    "PMT-1002": {"amount":  48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed",   "value_date": "2026-09-02", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held",     "value_date": "2026-09-02", "reason_code": "LIMIT_BREACH"},
    "PMT-1004": {"amount":   1200.00, "ccy": "GBP", "counterparty": "ACME-UK",
                 "status": "failed",   "value_date": "2026-09-03", "reason_code": "INVALID_IBAN"},
    "PMT-1005": {"amount": 750000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "held",     "value_date": "2026-09-03", "reason_code": "SANCTIONS_REVIEW"},
}

POLICY = {
    "INSUFFICIENT_FUNDS": "Retry once after 24h. If it fails again, notify the client desk. No manual funding.",
    "LIMIT_BREACH":       "Payments above USD 500,000 need Treasury approval before release.",
    "INVALID_IBAN":       "Return to originator with code R04. Never repair beneficiary details in-house.",
    "SANCTIONS_REVIEW":   "Hold. Compliance decides. Operations must not release or cancel.",
}

# Which reason codes may an agent resolve on its own, and which need a human?
NEEDS_HUMAN = {"LIMIT_BREACH", "SANCTIONS_REVIEW"}

print(f"{len(LEDGER)} payments, {len(POLICY)} policy rules loaded")

## Concept

A model call is a **function**: text in, text out, nothing retained. An **agent** is that call
placed inside a **loop**, where the output chooses the next action and the result is fed back.

Three things make the loop safe rather than merely clever:

| Piece | Question it answers |
|---|---|
| **State** | what has happened so far? |
| **Stop condition** | are we done, or out of budget? |
| **Loop detection** | are we going round without learning anything? |

The last two are what separate a demo from something you would run unattended.

## Section 1 &mdash; State is something you resend

The model has no memory, so *you* carry the conversation. `carry()` builds the full message
list for the next call: every earlier turn, then the new message.

In [ ]:
def carry(history, user_msg):
    """Build the message list for the next call.

    history: [(role, text), ...] of earlier turns, oldest first.
    Returns: [(role, text), ...] ending with the new human message.
    """
    msgs = []
    for role, text in history:      # the model gets the whole history back, every time
        msgs.append((role, text))
    msgs.append(("human", user_msg))
    return msgs

In [ ]:
# --- Self-check: Section 1
h = [("human", "The reference is PMT-1002."), ("ai", "Noted.")]
check("carry() replays every earlier turn", lambda: len(carry(h, "which reference?")) == 3)
check("carry() preserves the fact from turn 1",
      lambda: any("PMT-1002" in t for _, t in carry(h, "which reference?")),
      "the first turn must survive into the new call")
check("carry() puts the new message last",
      lambda: carry(h, "which reference?")[-1] == ("human", "which reference?"))

## Section 2 &mdash; The loop, and the budget that bounds it

`run_agent` is the whole agent: ask `decide` what to do, do it, record what happened, repeat.
It is `should_stop` that keeps it from running forever, so that is the part you write.

In [ ]:
MAX_STEPS = 6

def should_stop(state):
    """Return (stop, reason). Two reasons matter here: the goal, and the budget."""
    if state["answer"] is not None:
        return True, "goal"
    if state["steps"] >= state["max_steps"]:     # a hard number, not a hope
        return True, "budget"
    return False, None


def run_agent(goal, decide, tools, max_steps=MAX_STEPS):
    """decide(state) -> {"tool": name, "args": {...}}; the tool name "final" ends the run."""
    state = {"goal": goal, "steps": 0, "answer": None, "trace": [], "max_steps": max_steps}
    while True:
        stop, why = should_stop(state)
        if stop:
            state["stopped"] = why
            return state
        action = decide(state)
        if action["tool"] == "final":
            state["answer"] = action["args"]["text"]
            continue
        observation = tools[action["tool"]](**action["args"])
        state["trace"].append((action["tool"], action["args"], observation))
        state["steps"] = state["steps"] + 1

In [ ]:
# --- Self-check: Section 2   (a scripted `decide` -- no model involved, so this is deterministic)
_tools = {"peek": lambda ref: LEDGER.get(ref, {}).get("status", "unknown")}

def _finisher(state):
    if state["steps"] >= 2:
        return {"tool": "final", "args": {"text": "PMT-1002 failed: INSUFFICIENT_FUNDS"}}
    return {"tool": "peek", "args": {"ref": "PMT-1002"}}

def _never_finishes(state):
    return {"tool": "peek", "args": {"ref": "PMT-1002"}}

check("a run that reaches its goal stops with reason 'goal'",
      lambda: run_agent("g", _finisher, _tools)["stopped"] == "goal")
check("a run that never finishes stops on the budget",
      lambda: run_agent("g", _never_finishes, _tools)["stopped"] == "budget",
      "should_stop() must compare steps against max_steps")
check("the budget is actually respected",
      lambda: run_agent("g", _never_finishes, _tools)["steps"] == MAX_STEPS,
      "state['steps'] has to advance on every tool call")
check("the trace records every observation",
      lambda: len(run_agent("g", _finisher, _tools)["trace"]) == 2)

## Section 3 &mdash; Loop detection

A budget stops a runaway agent *eventually*. Loop detection stops it **as soon as it stops
learning** &mdash; the same tool, the same arguments, no new information. In production this is
usually the difference between a cheap failure and an expensive one.

In [ ]:
def is_looping(trace, window=3):
    """True when the last `window` tool calls are identical in both tool and arguments."""
    calls = [(tool, json.dumps(args, sort_keys=True)) for tool, args, _ in trace]
    if len(calls) < window:
        return False
    return len(set(calls[-window:])) == 1        # one distinct call across the window

In [ ]:
# --- Self-check: Section 3
_same = [("peek", {"ref": "PMT-1002"}, "failed")] * 3
_mixed = [("peek", {"ref": "PMT-1002"}, "failed"),
          ("peek", {"ref": "PMT-1003"}, "held"),
          ("peek", {"ref": "PMT-1002"}, "failed")]

check("three identical calls count as a loop", lambda: is_looping(_same) is True)
check("varied calls are not a loop", lambda: is_looping(_mixed) is False,
      "different arguments mean the agent is still learning something")
check("too short a trace is not yet a loop", lambda: is_looping(_same[:2]) is False)
check("the window is honoured", lambda: is_looping(_same, window=2) is True)

## Run it for real

Two calls, where the second depends on the first. Watch the model fail to recall &mdash; then watch
`carry()` fix it, by resending what it already told you.

In [ ]:
if llm_ready():
    print("--- without carry() -------------------------------------------")
    print("call 1:", ask("Remember this reference: PMT-1002. Reply with just OK."))
    print("call 2:", ask("Which payment reference did I just give you?"))

    print("\n--- with carry() ----------------------------------------------")
    history = [("human", "Remember this reference: PMT-1002."), ("ai", "OK")]
    try:
        msgs = carry(history, "Which payment reference did I just give you?")
        print("call 2:", get_llm().invoke(msgs).content)
    except NameError:
        print("(fill in carry() above, then re-run this cell)")
    except Exception as exc:
        print(f"<model unavailable: {type(exc).__name__}: {exc}>")

### Read it

The first pair shows the gap: call 2 has no access to call 1. The second pair shows the patch,
and the bill that comes with it &mdash; you resend the entire history on **every** turn. That is why
Module 3 spends its time on compaction rather than on bigger context windows.

In [ ]:
score()

## Your turn

1. Add a third stop reason to `should_stop`: **no path forward** &mdash; the last observation was an
   error and the agent has no untried tool. Which of the four stop conditions from the slides
   does that leave unimplemented?
2. `is_looping` compares arguments exactly. An agent that asks for `PMT-1002` then ` PMT-1002 `
   would slip past it. Normalise the arguments and decide, in a sentence, whether being strict
   or being lenient is the safer default here.